# Suite CNT — The container layout

`Container` gives each container shape a grid of cells before its children, then places each child it gains and, before each it loses is removed, closes the gap it leaves, with no coordinates of any kind. Each step is one mutation, a transaction expression the bench journals, and the container layout's adjustments to it, in order, which the step asserts; the suite's last case replays the journal in a fresh store, and finds the same adjustments. mbse-basis [CAP-238: container schemas and grid insertion](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-238.md), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
from support import Bench, text
from mbse.Diagrams.Framework import Container, Logical

b = Bench()

<a id="CNT-01"></a>
## CNT-01 · The install, after Logical's, defines the container layout's schemas, each valid, and run again writes nothing

In [ ]:
NAMES = ["Container.Layout", "Container.OfGrid", "Container.Cells", "Container.OfCell", "Container.Holds",
         "Container.Rules", "Logical.OfShape"]
schemas = [b.store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
b.store.transaction(Container.install)
assert [b.store.registered(name) for name in NAMES] == schemas
assert {"grid", "cell", "rules_from", "rules_to"} <= set(b.store.registered("Logical.OfShape").adjacencies)

<a id="CNT-02"></a>
## CNT-02 · A container is given its grid before its children, each of which then takes the next cell

In [ ]:
assert b.grid('cpu', 2) == '[{"kind":"grid","shape":"cpu","span":2,"grows":"rows","cell_rotation":0.0}]'
assert b.child('cpu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('cpu', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('cpu', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[1,0]},{"kind":"extents","rows":2,"columns":2}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[1,0],[["c",2]],null,1,1]]'
assert text([Container.cell_of(b.shape("cpu.b")), Container.cell_of(b.shape("cpu"))]) == '[[0,1],null]'

<a id="CNT-03"></a>
## CNT-03 · A child added takes the next cell, and the grid's extents grow only where it widens or lengthens

In [ ]:
assert b.child('cpu', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,1]}]'
assert b.child('cpu', 'e') == '[{"kind":"place","shape":"e","order":4,"to":[2,0]},{"kind":"extents","rows":3,"columns":2}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[1,0],[["c",2]],null,1,1],[[1,1],[["d",3]],null,1,1],[[2,0],[["e",4]],null,1,1]]'
assert b.grid('one', 3) == '[{"kind":"grid","shape":"one","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('one', 'x') == '[{"kind":"place","shape":"x","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'

<a id="CNT-04"></a>
## CNT-04 · A grid that grows columns fills each column first

In [ ]:
assert b.grid('bus', 2, 'columns') == '[{"kind":"grid","shape":"bus","span":2,"grows":"columns","cell_rotation":0.0}]'
assert b.child('bus', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('bus', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[1,0]},{"kind":"extents","rows":2,"columns":1}]'
assert b.child('bus', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,1]},{"kind":"extents","rows":2,"columns":2}]'
assert b.child('bus', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,1]}]'
assert b.child('bus', 'e') == '[{"kind":"place","shape":"e","order":4,"to":[0,2]},{"kind":"extents","rows":2,"columns":3}]'
assert b.cells("bus") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[0,2],[["e",4]],null,1,1],[[1,0],[["b",1]],null,1,1],[[1,1],[["d",3]],null,1,1]]'

<a id="CNT-05"></a>
## CNT-05 · A child about to be removed leaves its cell, and the children after it move back one each

In [ ]:
assert b.unchild('cpu', 'b') == '[{"kind":"leave","shape":"b","from":[0,1]},{"kind":"move","shape":"c","from":[1,0],"to":[0,1]},{"kind":"move","shape":"d","from":[1,1],"to":[1,0]},{"kind":"move","shape":"e","from":[2,0],"to":[1,1]},{"kind":"extents","rows":2,"columns":2}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'
assert b.unchild('cpu', 'e') == '[{"kind":"leave","shape":"e","from":[1,1]}]'
assert b.unchild('bus', 'a', 'd') == '[{"kind":"leave","shape":"a","from":[0,0]},{"kind":"move","shape":"b","from":[1,0],"to":[0,0]},{"kind":"move","shape":"c","from":[0,1],"to":[1,0]},{"kind":"move","shape":"d","from":[1,1],"to":[0,1]},{"kind":"move","shape":"e","from":[0,2],"to":[1,1]},{"kind":"extents","rows":2,"columns":2},{"kind":"leave","shape":"d","from":[0,1]},{"kind":"move","shape":"e","from":[1,1],"to":[0,1]}]'
assert b.cells("bus") == '[[[0,0],[["b",1]],null,1,1],[[0,1],[["e",4]],null,1,1],[[1,0],[["c",2]],null,1,1]]'
assert b.unchild('one', 'x') == '[{"kind":"leave","shape":"x","from":[0,0]},{"kind":"extents","rows":0,"columns":0}]'
assert b.cells("one") == '[]'
assert text(b.problems("cpu", "bus", "one")) == '[[],[],[]]'

<a id="CNT-06"></a>
## CNT-06 · What a grid refuses: an axis it has not, a span below 1, a second grid, a container with children, and a child of a container without one

In [ ]:
assert b.grid('diagonal', 2, 'diagonal') == "ValueError: a grid grows rows or columns, not 'diagonal'"
assert b.grid('empty', 0) == "ValueError: a grid's span is a positive int, not 0"
assert b.grid('cpu', 2) == 'ValueError: a shape has one grid'
b.logical(["kids.x"])
assert b.grid('kids', 2) == "ValueError: a container's grid comes before its children, which reach it as mutations"
assert b.child('bare', 'y') == 'ValueError: a container without a grid has no container layout: give it one, with layout'

<a id="CNT-07"></a>
## CNT-07 · The journal, replayed in a fresh store, makes the same adjustments

In [ ]:
assert text(b.replay()) == '[]'